# MIT-BIH Arrhythmia

En esta notebook, implementaremos técnicas de regularización, tales como early stopping y dropout, utilizando el [MIT-BIH Arrhythmia Database](https://physionet.org/content/mitdb/1.0.0). El objetivo principal es mejorar la generalización de nuestros modelos de aprendizaje profundo y prevenir el sobreajuste durante el entrenamiento.

## Introducción

### Objetivos

1. **Implementar y entender las técnicas de regularización** más comunes en redes neuronales, incluyendo early stopping y dropout.
2. **Entrenar modelos de clasificación** en PyTorch utilizando el MIT-BIH Arrhythmia Database.
3. **Evaluar el impacto de las técnicas de regularización** en el rendimiento del modelo utilizando métricas adecuadas.

### Contenido

1. Configuración de bibliotecas y semillas para reproducibilidad.
2. Carga y exploración del MIT-BIH Arrhythmia Database.
3. Preparación de los datos y división en conjuntos de entrenamiento, validación y prueba.
4. Definición y entrenamiento de un modelo de clasificación en PyTorch: logits, `CrossEntropyLoss` y métricas de clasificación.
5. Implementación de técnicas de regularización: dropout, early stopping y weight decay.
6. Comparación de los modelos en validación y evaluación final en prueba.

### Sobre el conjunto de datos

El MIT-BIH Arrhythmia Database es un conjunto de datos ampliamente utilizado en la investigación de la arritmia cardíaca. Contiene registros de electrocardiogramas (ECG) de diferentes pacientes, con anotaciones detalladas sobre distintos tipos de arritmias. Este conjunto de datos nos permitirá entrenar modelos de clasificación capaces de identificar distintos tipos de latidos a partir de la señal de ECG, haciendo énfasis en la importancia de la regularización para mejorar la generalización de los modelos.

Usamos la versión ya preprocesada que se distribuye en Kaggle: [ECG Heartbeat Categorization Dataset](https://www.kaggle.com/shayanfazeli/heartbeat), preparada por Kachuee, Fazeli y Sarrafzadeh para el paper [*ECG Heartbeat Classification: A Deep Transferable Representation* (2018)](https://arxiv.org/abs/1805.00794).


In [ ]:
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from torchinfo import summary

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    recall_score,
    ConfusionMatrixDisplay,
)

from utils import get_device, get_num_workers, plot_training


In [ ]:
# Fijamos la semilla para que los resultados sean reproducibles
SEED = 34

torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True  # cuDNN tiene varias implementaciones de cada operación (convoluciones, etc.); esto fuerza las que dan siempre el mismo resultado, a costa de velocidad


In [ ]:
DEVICE = get_device()  # cuda > mps > xpu > cpu
# Linux: mitad de los núcleos disponibles, entre 1 y 8; Windows/macOS: 0 (ver docstring)
NUM_WORKERS = get_num_workers()

print(f"Usando {DEVICE}")
print(f"Usando {NUM_WORKERS}")


In [ ]:
# Batch chico a propósito: con 256 muestras por batch, cada época hace ~274 actualizaciones de los pesos
# (serían 35 con un batch de 2048). Más actualizaciones por época = el modelo ajusta más rápido, y también
# memoriza antes el conjunto de entrenamiento, que es justamente lo que queremos observar en esta clase
BATCH_SIZE = 256


## Carga de datos + Exploración

Los datos no están en el repositorio. Descargar los dos CSV desde Kaggle (requiere cuenta) y dejarlos en la carpeta `data/` junto a esta notebook. Con la [CLI de Kaggle](https://github.com/Kaggle/kaggle-api) configurada (viene en el `environment.yml`):

```bash
kaggle datasets download -d shayanfazeli/heartbeat -p data --unzip
```

Solo usamos `mitbih_train.csv` y `mitbih_test.csv` (los archivos `ptbdb_*` son otro dataset del mismo paquete).


In [ ]:
TRAIN_DATA_PATH = "data/mitbih_train.csv"
TEST_DATA_PATH = "data/mitbih_test.csv"


In [ ]:
# header=None: los CSV no tienen fila de encabezado, las columnas quedan numeradas 0..187
df_train = pd.read_csv(TRAIN_DATA_PATH, header=None)
df_test = pd.read_csv(TEST_DATA_PATH, header=None)

ninputs = df_train.shape[1] - 1  # todas las columnas menos la última (la etiqueta)
nclasses = df_train.iloc[:, -1].nunique()
print(f"Train: {df_train.shape[0]} latidos | Test: {df_test.shape[0]} latidos")
print(f"Existen {nclasses} clases y {ninputs} características")


In [ ]:
df_train.info()
df_train.head()


### ¿Qué es cada fila?

Cada fila es **un latido**, no un paciente ni un registro completo. Los autores del dataset tomaron las señales de ECG del MIT-BIH (360 Hz), detectaron cada pico R, recortaron una ventana alrededor de cada latido, la remuestrearon a **125 Hz** y la escalaron para que la amplitud quede en $[0, 1]$. Por eso:

- Las **187 columnas** son 187 muestras consecutivas de la señal (1.5 segundos a 125 Hz). Es una serie temporal, no 187 variables clínicas.
- Los **ceros al final** de cada fila son relleno (*zero-padding*): los latidos tienen distinta duración y se completaron con ceros hasta 187.
- Los valores ya están entre 0 y 1, latido por latido. **En esta clase no estandarizamos las entradas** como en la clase 02: el preprocesamiento del dataset ya lo hizo, y como se hizo por latido (con el máximo de cada uno) no hay estadísticos de train que filtrar hacia test.

La **última columna** es la etiqueta, un entero de 0 a 4. Sigue el estándar AAMI de agrupación de latidos:

| Código | Clase | Qué incluye |
|---|---|---|
| 0 | **N** – Normal beat | Latidos normales y bloqueos de rama (conducción normal desde el nodo sinusal). |
| 1 | **S** – Supraventricular premature beat | Latidos prematuros que nacen arriba de los ventrículos (aurículas, nodo AV). |
| 2 | **V** – Premature ventricular contraction | Latidos prematuros que nacen en los ventrículos; forma ancha y distinta. |
| 3 | **F** – Fusion of ventricular and normal beat | Mezcla de un latido normal y uno ventricular que ocurren casi a la vez. |
| 4 | **Q** – Unclassifiable beat | Latidos de marcapasos, fusiones con marcapasos y no clasificables. Por eso es una clase grande. |


In [ ]:
# separamos características (todas las columnas menos la última) y etiquetas (última columna)
X_train_full = df_train.iloc[:, :-1]
y_train_full = df_train.iloc[:, -1].astype(int)

X_test = df_test.iloc[:, :-1]
y_test = df_test.iloc[:, -1].astype(int)


In [ ]:
# nombres de las clases, en el orden de los códigos 0..4
TARGET_NAMES = [
    "Normal beat",
    "Supraventricular premature beat",
    "Premature ventricular contraction",
    "Fusion of ventricular and normal beat",
    "Unclassifiable beat",
]


Miremos un latido de cada clase. El eje x es tiempo (muestras a 125 Hz, 8 ms cada una) y el eje y la amplitud escalada. Se ve el pico R al principio, la forma de cada tipo de latido y el relleno con ceros al final.


In [ ]:
fig, axes = plt.subplots(1, nclasses, figsize=(20, 3.5), sharey=True)
for k, ax in enumerate(axes):
    idx = y_train_full[y_train_full == k].index[0]  # primer latido de la clase k
    ax.plot(X_train_full.loc[idx].values)
    ax.set_title(f"{k}: {TARGET_NAMES[k]}", fontsize=9)
    ax.set_xlabel("muestra (125 Hz)")
axes[0].set_ylabel("amplitud (escalada)")
plt.tight_layout()
plt.show()


### Distribución de clases

Veamos la distribución de las clases en train y en test para comprender mejor el problema de clasificación que estamos abordando. Graficamos **proporciones** y no cantidades, para poder comparar los dos conjuntos aunque tengan distinto tamaño.


In [ ]:
class_dist = pd.DataFrame(
    {
        "train": y_train_full.value_counts(normalize=True).sort_index(),
        "test": y_test.value_counts(normalize=True).sort_index(),
    }
)
class_dist.index = [f"{k}: {name}" for k, name in enumerate(TARGET_NAMES)]
print(class_dist.round(4))

class_dist.plot(kind="bar", figsize=(10, 4), title="Proporción de latidos por clase", rot=20)
plt.ylabel("proporción")
plt.show()

majority = y_train_full.value_counts(normalize=True).max()
print(f"\nLa clase mayoritaria (Normal) es el {majority:.1%} de train")


Las clases están **muy desbalanceadas**: un modelo que responda siempre "Normal" acierta el 82.8% de las veces sin aprender nada. Ese es el número a superar, y por eso la *accuracy* sola no alcanza para evaluar este problema: vamos a mirar métricas por clase (precision, recall, F1) y su promedio **macro**, que pesa igual a todas las clases.

Train y test tienen la misma distribución de clases (los autores del dataset hicieron el split estratificado), así que el conjunto de prueba es representativo.

> Existen estrategias para atacar el desbalance durante el entrenamiento, como dar más peso a las clases minoritarias en la pérdida (`nn.CrossEntropyLoss(weight=...)`) o sobremuestrearlas. Quedan para el ejercicio libre.


## Datasets y Dataloaders

### Dataset

Vamos a crear un `Dataset` personalizado para cargar los datos del MIT-BIH Arrhythmia Database desde nuestros DataFrames de pandas y devolverlos en el formato adecuado para ser procesados por nuestros modelos de PyTorch.

A diferencia de la clase 02 (regresión), la etiqueta es un **índice de clase** y se devuelve como entero (`torch.long`) con shape `()`, no como float con shape `(1,)`. Es lo que espera la función de pérdida de clasificación que usamos más abajo.


In [ ]:
class MITBIHDataSet(Dataset):
    def __init__(self, df_features, df_target):
        self.x = df_features.to_numpy()  # indexar un array de numpy es más rápido que un DataFrame
        self.y = df_target.to_numpy()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x = torch.tensor(self.x[idx], dtype=torch.float32)  # shape (187,)
        y = torch.tensor(self.y[idx], dtype=torch.long)  # índice de clase, shape ()
        return x, y


### Split de datos

Separamos una parte del conjunto de entrenamiento para validación. Esta vez, en vez de hacerlo con [torch.utils.data.random_split](https://pytorch.org/docs/stable/data.html#torch.utils.data.random_split), lo haremos con [sklearn.model_selection.train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) con `stratify=y`: el split se hace **clase por clase**, de modo que cada clase queda con la misma proporción en train y en validación. Con un split aleatorio simple, una clase con 640 latidos (la 3) podría quedar mal repartida.


In [ ]:
# Dividir los datos de entrenamiento en entrenamiento y validación (80/20), manteniendo la proporción de clases
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=SEED, stratify=y_train_full
)

# verificamos que las proporciones de clase coinciden
pd.DataFrame(
    {
        "train": y_train.value_counts(normalize=True).sort_index(),
        "val": y_val.value_counts(normalize=True).sort_index(),
    }
).round(4)


In [ ]:
train_dataset = MITBIHDataSet(X_train, y_train)
val_dataset = MITBIHDataSet(X_val, y_val)
test_dataset = MITBIHDataSet(X_test, y_test)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")


### DataLoaders

Definimos los dataloaders para cada conjunto de datos, estos son los que se encargan de cargar los datos en lotes durante el entrenamiento y la evaluación del modelo.


In [ ]:
def get_data_loaders(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers
    )  # shuffle=True solo en train

    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )

    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )

    return train_loader, val_loader, test_loader


In [ ]:
train_loader, val_loader, test_loader = get_data_loaders()

# probamos un batch del DataLoader
x_batch, y_batch = next(iter(train_loader))
print(x_batch.shape, x_batch.dtype)  # [256, 187] float32
print(y_batch.shape, y_batch.dtype)  # [256] int64: un índice de clase por muestra (no [256, 1] como en regresión)


## Definición de modelo

Empezaremos definiendo un modelo de clasificación simple en PyTorch: un MLP con **dos capas ocultas** (512 y 2048 neuronas, con ReLU) y una capa de salida con **una neurona por clase**. Es deliberadamente grande para el problema (más de un millón de parámetros para 187 entradas): queremos que sobreajuste, para tener algo que regularizar.

La capa de salida **no lleva activación**. Explicamos por qué a continuación.


In [ ]:
# MLP                                      [256, 5]
# ├─Linear: 1-1                            [256, 512]
# ├─Linear: 1-2                            [256, 2048]
# ├─Linear: 1-3                            [256, 5]


class MLP(nn.Module):
    def __init__(self, input_size, nclass):
        super(MLP, self).__init__()  # obligatorio: inicializa nn.Module para que registre capas y parámetros
        # TODO: definir tres capas lineales: input_size -> 512 -> 2048 -> nclass
        pass

    def forward(self, x):
        # TODO: aplicar las capas en orden con F.relu entre ellas. La última NO lleva activación:
        #       devuelve los logits, shape (batch, nclass)
        pass


summary(MLP(ninputs, nclasses), input_size=(BATCH_SIZE, ninputs))


### Logits, softmax y `CrossEntropyLoss`

En la clase 02 el modelo devolvía un número (el precio) y la pérdida era `MSELoss`. Acá el modelo devuelve **5 números por muestra**, uno por clase, sin acotar: se llaman **logits**. Para convertirlos en probabilidades se aplica la función **softmax**, que los exponencia y normaliza para que sumen 1:

$$ p_k = \frac{e^{z_k}}{\sum_j e^{z_j}} $$

La pérdida para clasificación es la **entropía cruzada**: $-\log p_{y}$, menos el logaritmo de la probabilidad que el modelo le asignó a la clase correcta $y$. Vale 0 si el modelo le da probabilidad 1 a la clase correcta y crece sin límite a medida que esa probabilidad baja. Esto es lo que la distingue del MSE: castiga mucho más estar seguro y equivocado.

En PyTorch, [`nn.CrossEntropyLoss`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) **incluye la softmax** (calcula `log_softmax` y después la pérdida, que es numéricamente más estable). Por eso:

- la capa de salida del modelo **no lleva softmax**: entrega logits;
- la etiqueta es un **índice de clase** (`long`, shape `(N,)`), no un vector one-hot;
- para predecir, alcanza con `argmax` de los logits: la softmax es monótona, así que el logit más grande es también la probabilidad más grande.


In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0, 0.1, -0.5]])  # salida inventada del modelo para una muestra (5 clases)
y_true = torch.tensor([0])  # la clase correcta es la 0

probs = F.softmax(logits, dim=1)
print("softmax:", probs.numpy().round(4), "| suma:", probs.sum().item())

loss = F.cross_entropy(logits, y_true)  # lo mismo que nn.CrossEntropyLoss()(logits, y_true)
print(f"cross_entropy: {loss.item():.4f} | -log(p_correcta): {-torch.log(probs[0, 0]).item():.4f}")

print("argmax de los logits:", logits.argmax(dim=1).item(), "| argmax de la softmax:", probs.argmax(dim=1).item())

# si la clase correcta fuera la 2 (p = 0.02), la pérdida sería mucho mayor:
print(f"cross_entropy si y=2: {F.cross_entropy(logits, torch.tensor([2])).item():.4f}")


## Entrenamiento

Reutilizamos las funciones `evaluate` y `train` de la clase 02, con una diferencia: ahora reciben el dispositivo como argumento en lugar de leer la variable global.


In [ ]:
def evaluate(model, criterion, data_loader, device):
    """
    Evalúa el modelo en los datos proporcionados y calcula la pérdida promedio.

    Args:
        model (torch.nn.Module): El modelo que se va a evaluar.
        criterion (torch.nn.Module): La función de pérdida que se utilizará para calcular la pérdida.
        data_loader (torch.utils.data.DataLoader): DataLoader que proporciona los datos de evaluación.
        device (str): El dispositivo donde se ejecutará la evaluación.

    Returns:
        float: La pérdida promedio en el conjunto de datos de evaluación.

    """
    model.eval()  # ponemos el modelo en modo de evaluacion
    total_loss = 0  # acumulador de la perdida
    with torch.no_grad():  # deshabilitamos el calculo de gradientes
        for x, y in data_loader:  # iteramos sobre el dataloader
            x = x.to(device)  # movemos los datos al dispositivo
            y = y.to(device)  # movemos los datos al dispositivo
            output = model(x)  # forward pass
            total_loss += criterion(output, y).item()  # acumulamos la perdida
    return total_loss / len(data_loader)  # retornamos la perdida promedio


def train(
    model,
    optimizer,
    criterion,
    train_loader,
    val_loader,
    device,
    epochs=10,
    log_fn=None,
    log_every=1,
):
    """
    Entrena el modelo utilizando el optimizador y la función de pérdida proporcionados.

    Args:
        model (torch.nn.Module): El modelo que se va a entrenar.
        optimizer (torch.optim.Optimizer): El optimizador que se utilizará para actualizar los pesos del modelo.
        criterion (torch.nn.Module): La función de pérdida que se utilizará para calcular la pérdida.
        train_loader (torch.utils.data.DataLoader): DataLoader que proporciona los datos de entrenamiento.
        val_loader (torch.utils.data.DataLoader): DataLoader que proporciona los datos de validación.
        device (str): El dispositivo donde se ejecutará el entrenamiento.
        epochs (int): Número de épocas de entrenamiento (default: 10).
        log_fn (function): Función que se llamará después de cada log_every épocas con los argumentos (epoch, train_loss, val_loss) (default: None).
        log_every (int): Número de épocas entre cada llamada a log_fn (default: 1).

    Returns:
        Tuple[List[float], List[float]]: Una tupla con dos listas, la primera con el error de entrenamiento de cada época y la segunda con el error de validación de cada época.

    """
    epoch_train_errors = []  # colectamos el error de training para posterior analisis
    epoch_val_errors = []  # colectamos el error de validacion para posterior analisis

    for epoch in range(epochs):  # loop de entrenamiento
        model.train()  # ponemos el modelo en modo de entrenamiento
        train_loss = 0  # acumulador de la perdida de entrenamiento
        for x, y in train_loader:
            x = x.to(device)  # movemos los datos al dispositivo
            y = y.to(device)  # movemos los datos al dispositivo

            optimizer.zero_grad()  # reseteamos los gradientes

            output = model(x)  # forward pass (prediccion)
            batch_loss = criterion(output, y)  # calculamos la perdida con la salida esperada

            batch_loss.backward()  # backpropagation
            optimizer.step()  # actualizamos los pesos

            train_loss += batch_loss.item()  # acumulamos la perdida

        train_loss /= len(train_loader)  # calculamos la perdida promedio de la epoca
        epoch_train_errors.append(train_loss)  # guardamos la perdida de entrenamiento
        val_loss = evaluate(model, criterion, val_loader, device)  # evaluamos el modelo en el conjunto de validacion
        epoch_val_errors.append(val_loss)  # guardamos la perdida de validacion

        if log_fn is not None:  # si se pasa una funcion de log
            if (epoch + 1) % log_every == 0:  # loggeamos cada log_every epocas
                log_fn(epoch, train_loss, val_loss)  # llamamos a la funcion de log

    return epoch_train_errors, epoch_val_errors


Hiperparámetros. Entrenamos **más épocas de las necesarias a propósito**: queremos ver el sobreajuste en la curva de validación para después atacarlo con las técnicas de regularización.

> Usamos el learning rate por defecto de Adam, `1e-3`. Con un valor mayor (por ejemplo `0.01`) el modelo converge más rápido, pero la pérdida de validación queda muy ruidosa y eso confunde a las técnicas que siguen. Probarlo es parte del ejercicio 5.


In [ ]:
LR = 0.001
CRITERION = nn.CrossEntropyLoss()  # incluye la softmax; espera logits (N, C) y etiquetas long (N,)
EPOCHS = 100


In [ ]:
def print_log(epoch, train_loss, val_loss):
    print(
        f"Epoch: {epoch + 1:03d}/{EPOCHS:03d} | Train Loss: {train_loss:.5f} | Val Loss: {val_loss:.5f}"
    )


# guardamos las curvas y las métricas de cada modelo para compararlos al final
histories = {}
val_metrics = {}

torch.manual_seed(SEED)  # misma inicialización para todos los modelos que entrenemos: la comparación es justa
base_model = MLP(ninputs, nclasses).to(DEVICE)

optimizer = optim.Adam(base_model.parameters(), lr=LR)

histories["base"] = train(
    model=base_model,
    optimizer=optimizer,
    criterion=CRITERION,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    epochs=EPOCHS,
    log_fn=print_log,
    log_every=5,
)


## Loss durante el entrenamiento

La línea punteada marca la época con menor pérdida de validación. Hasta ahí el modelo generaliza mejor con cada época; a partir de ahí la pérdida de entrenamiento sigue bajando hacia cero pero la de validación **sube**, y termina bastante por encima de su mínimo: el modelo **memoriza** el conjunto de entrenamiento. Eso es el sobreajuste, y es lo que las tres técnicas que siguen intentan evitar.

Un detalle que vamos a retomar al final: en esta etapa las **predicciones** del modelo (el `argmax`) cambian poco; lo que crece es la **confianza** con la que se equivoca. La entropía cruzada castiga eso (recordar el ejemplo de $-\log p$), la accuracy no lo ve. Por eso seguimos la pérdida de validación y no solo la accuracy.


In [ ]:
plot_training(*histories["base"], mark_min=True, title="Modelo base")


## Evaluación

Vamos a evaluar el modelo con métricas de clasificación por clase. Antes, un resumen de qué significa cada una. En un problema multiclase, scikit-learn calcula precision, recall y F1 **para cada clase por separado**, tratándola como "esta clase contra el resto" (TP, FP y FN se cuentan respecto de esa clase), y después promedia.

**Métricas por Clase**
- **Precision:** De lo que predije como esta clase, ¿cuánto acerté? (↑ = menos falsos positivos)
  - `Precision = TP / (TP + FP)`
- **Recall:** De todos los casos reales, ¿cuántos detecté? (↑ = menos falsos negativos)
  - `Recall = TP / (TP + FN)`
- **F1-Score:** Balance entre precision y recall (media armónica)
  - `F1 = 2 * (Precision * Recall) / (Precision + Recall)`
- **Support:** Cantidad de muestras reales de esa clase

**Métricas Globales**
- **Accuracy:** % total de aciertos `(TP + TN) / Total`
- **Macro Avg:** Promedio simple (todas las clases pesan igual)
- **Weighted Avg:** Promedio ponderado por support (refleja desbalance)

**Interpretación Rápida**
- Valores → 1.0 = Mejor | Valores → 0.0 = Peor
- Precision > Recall = Modelo conservador
- Recall > Precision = Modelo agresivo
- Macro < Weighted = Mejor en clases mayoritarias
- Support desigual = Dataset desbalanceado

> **Donde:** TP = True Positives, FP = False Positives, FN = False Negatives, TN = True Negatives

**¿En qué conjunto evaluamos?** En esta clase vamos a entrenar varios modelos y elegir uno. Esa elección se hace con el conjunto de **validación**; el de **prueba** se usa una sola vez, al final, con el modelo elegido. Si eligiéramos mirando test, el número final estaría inflado: habríamos "entrenado" la elección con los datos de prueba.


In [ ]:
def model_classification_report(model, dataloader, device, target_names):
    '''
    Imprime accuracy y el reporte de clasificación, y devuelve (etiquetas, predicciones) para seguir analizando.
    '''
    # Evaluación del modelo
    model.eval()

    all_preds = []
    all_labels = []

    # TODO:
    #   1. sin calcular gradientes, recorrer el dataloader
    #   2. mover las entradas al dispositivo y obtener los logits del modelo
    #   3. la predicción es el argmax de los logits en la dimensión de las clases (dim=1)
    #   4. acumular predicciones y etiquetas en las listas (en CPU, como numpy)
    pass

    # Calcular precisión (accuracy)
    accuracy = accuracy_score(all_labels, all_preds)
    print(f"Accuracy: {accuracy:.4f}\n")

    # Reporte de clasificación
    report = classification_report(all_labels, all_preds, target_names=target_names)
    print("Reporte de clasificación:\n", report)

    return all_labels, all_preds


Además del reporte, graficamos la **matriz de confusión**: fila = clase real, columna = clase predicha, normalizada por fila (cada fila suma 1, así que la diagonal es el recall de cada clase). Muestra *con qué* se confunde cada clase, cosa que el reporte no dice.

Definimos una función auxiliar que hace las dos cosas y guarda las métricas macro en `val_metrics`, porque la vamos a usar con cada modelo.


In [ ]:
def evaluate_model(name, model, dataloader, title=None):
    labels, preds = model_classification_report(model, dataloader, DEVICE, TARGET_NAMES)

    val_metrics[name] = {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "recall_macro": recall_score(labels, preds, average="macro"),
        "val_loss": evaluate(model, CRITERION, dataloader, DEVICE),  # pérdida con los pesos finales (o restaurados)
    }

    short_names = ["N", "S", "V", "F", "Q"]
    ConfusionMatrixDisplay.from_predictions(
        labels, preds, display_labels=short_names, normalize="true", values_format=".2f", cmap="Blues"
    )
    plt.title(title or f"Matriz de confusión ({name}, validación)")
    plt.show()


evaluate_model("base", base_model, val_loader)


La accuracy es alta, pero el reporte muestra el efecto del desbalance: las clases **S** y **F** (las más chicas) tienen el peor recall, y en la matriz de confusión se ve que se confunden sobre todo con **N**. El F1 macro es bastante menor que la accuracy: ese es el número que vamos a usar para comparar modelos.


## Ejercicios

1. **Implementar dropout**: Agregar dropout después de cada capa oculta del modelo. Evaluar el impacto de dropout en el rendimiento del modelo.

2. **Implementar early stopping**: Implementar la técnica de early stopping para detener el entrenamiento cuando el rendimiento del modelo deja de mejorar en el conjunto de validación. Evaluar el impacto de early stopping en el rendimiento del modelo.

3. **Aplicar weight decay**: Entrenar el modelo con regularización L2 a través del parámetro `weight_decay` del optimizador. Evaluar el impacto en el rendimiento y en la magnitud de los pesos.

4. **Comparación de modelos**: Con los resultados en **validación**, ¿con cuál de los modelos te quedarías si:
    - Tuvieras que elegir el modelo con mejor precisión en promedio?
    - Tuvieras que elegir el modelo con mejor recall en promedio?
    - Tuvieras que elegir el modelo con mejor F1-score en promedio?

5. **Modelo libre**: Experimentar con diferentes arquitecturas, hiperparámetros y técnicas de regularización para mejorar el rendimiento. Algunas ideas: `BATCH_SIZE=2048` o `EPOCHS=200` (menos o más sobreajuste en el modelo base), `lr=0.01` (observar qué pasa con las curvas y con early stopping), `p=0.5` en dropout o `weight_decay=1e-3` (regularización excesiva), combinar las tres técnicas, label smoothing (`nn.CrossEntropyLoss(label_smoothing=0.1)`), o pesar las clases minoritarias en la pérdida con `nn.CrossEntropyLoss(weight=...)`.


### Ejercicio 1: Implementar dropout

[Dropout](https://jmlr.org/papers/v15/srivastava14a.html) es una técnica de regularización utilizada en redes neuronales para prevenir el sobreajuste (overfitting). Consiste en desactivar aleatoriamente un subconjunto de neuronas durante el entrenamiento en cada paso de propagación hacia adelante. Esto ayuda a la red a no depender demasiado de ninguna neurona en particular, promoviendo una representación más robusta y generalizable de los datos.

**¿Cómo Funciona Dropout?**

Durante el entrenamiento, Dropout:
1. **Apagado Aleatorio de Neuronas**: Cada neurona se apaga (su salida se pone en 0) con probabilidad `p`, de forma independiente, en cada paso.
2. **Escalado de Neuronas Activas**: Las salidas de las neuronas que quedaron activas se multiplican por `1 / (1 - p)`. Así, el valor esperado de cada activación es el mismo con y sin dropout, y la red no ve una escala distinta en entrenamiento y en evaluación.

**Evaluación y Dropout**

Durante la evaluación, es crucial que las capas Dropout se comporten de manera diferente:
- **Entrenamiento (`model.train()`)**: Dropout está activo: apaga neuronas y escala las restantes.
- **Evaluación (`model.eval()`)**: Dropout está desactivado: la capa deja pasar todo tal cual, sin apagar ni escalar. El modelo usa toda su capacidad para predecir.

**Implementación de Dropout en PyTorch**

En PyTorch, podemos implementar Dropout utilizando la capa [`torch.nn.Dropout`](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html). Veámosla en acción con un tensor de ejemplo: con `p=0.5`, los valores que sobreviven aparecen multiplicados por 2.


In [ ]:
rand_torch = torch.rand((2, 5))  # genero datos al azar
print(rand_torch)  # imprimimos sus valores
drop_layer = nn.Dropout(0.5)  # creamos una capa de dropout con un 50% de apagar una neurona


In [ ]:
drop_layer.train()  # por defecto ya esta en train mode
# 1) apaga (pone en 0) cada valor con probabilidad p  2) multiplica los restantes por 1 / (1 - p) = 2
print(drop_layer(rand_torch))


In [ ]:
drop_layer.eval()  # en eval la capa no hace nada: la salida es idéntica a la entrada
print(drop_layer(rand_torch))


Algunas reglas prácticas para usar Dropout de manera efectiva:

- **Ubicación: "Después de Activación, Nunca en Output"**
```python
x = F.relu(self.fc(x))
x = self.dropout(x)  # ✅ Después de activación
# ❌ NUNCA dropout en la capa final
```

- **Valores: "20-50 Rule"**
```python
dropout = nn.Dropout(0.5)    # Capas ocultas (default)
dropout = nn.Dropout(0.2)    # CNNs, RNNs, entrada
# Si dudas → usa 0.3
```

- **Train/Eval: "Siempre Cambia Modo"**
```python
model.train()   # Entrenamiento
model.eval()    # ⚠️ CRÍTICO para inferencia
```

- **Cuándo Usar: "Solo si Overfittea"**
```python
# train_acc=99%, val_acc=85% → Agrega dropout
# train_acc=80%, val_acc=78% → No necesitas
# Empieza sin dropout, agrega si necesitas
```

- **Arquitectura: "Más Profundo = Menos Dropout"**
```python
# 2-3 capas:  p=0.5
# 5-10 capas: p=0.2-0.3
# 10+ capas:  p=0.1 o nada
```


In [ ]:
class MLP_EJ1(nn.Module):
    def __init__(self, input_size, nclass, dropout=0.5):
        super(MLP_EJ1, self).__init__()
        # TODO: las mismas tres capas lineales del MLP más una capa nn.Dropout(dropout)
        #       (una sola instancia alcanza: no tiene parámetros y se puede reutilizar)
        pass

    def forward(self, x):
        # TODO: igual que MLP, aplicando dropout después de cada ReLU. Nunca después de la última capa
        pass


summary(MLP_EJ1(ninputs, nclasses), input_size=(BATCH_SIZE, ninputs))


Dos preguntas para pensar, con sus respuestas:

- **¿Por qué dropout no tiene parámetros entrenables?** Porque no calcula nada a partir de pesos: solo decide al azar qué valores anular y multiplica el resto por una constante. En el `summary` aparece con `--` en la columna de parámetros.
- **¿Qué pasaría si no ponemos el modelo en `eval()` durante la evaluación?** Dropout seguiría activo: cada pasada por el modelo daría una predicción distinta (aleatoria) y con parte de las neuronas apagadas. Las métricas serían peores y no reproducibles.


In [ ]:
torch.manual_seed(SEED)  # misma inicialización que el modelo base
ej1_model = MLP_EJ1(ninputs, nclasses, dropout=0.2).to(DEVICE)

optimizer = optim.Adam(ej1_model.parameters(), lr=LR)

histories["dropout"] = train(
    model=ej1_model,
    optimizer=optimizer,
    criterion=CRITERION,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    epochs=EPOCHS,
    log_fn=print_log,
    log_every=5,
)


In [ ]:
plot_training(*histories["dropout"], mark_min=True, title="Dropout p=0.2")


In [ ]:
evaluate_model("dropout", ej1_model, val_loader)


**¿Cómo se compara con el modelo sin dropout?** Dos cosas para observar:

- En la curva, la pérdida de validación sube mucho menos que en el modelo base después del mínimo (comparar dónde termina cada una): la red tarda más en memorizar y memoriza menos. Dropout atenúa el sobreajuste, no lo elimina. Notar también que la pérdida de entrenamiento se mide **con dropout activo** (la red trabaja con neuronas apagadas), así que no es directamente comparable con la del modelo base.
- En las métricas, comparar el **F1 macro** y el recall de las clases minoritarias (S y F) con los del modelo base. Regularizar no es gratis: si el regularizador es demasiado fuerte para el problema (probar `p=0.5`), el modelo deja de aprender las clases raras (subajuste). El valor de `p` es un hiperparámetro más, y hay que elegirlo mirando validación.


#### Lectura Adicional
- [Dropout: A Simple Way to Prevent Neural Networks from Overfitting](https://jmlr.org/papers/v15/srivastava14a.html) - Paper original de Dropout
- [Deep Learning Book - Regularization](https://www.deeplearningbook.org/contents/regularization.html)
- [Dive into Deep Learning - Dropout](https://d2l.ai/chapter_multilayer-perceptrons/dropout.html)


### Ejercicio 2: Implementar early stopping

[Early stopping](https://en.wikipedia.org/wiki/Early_stopping) es una técnica de regularización utilizada para prevenir el sobreajuste en modelos de aprendizaje profundo. Consiste en detener el entrenamiento del modelo cuando el rendimiento en el conjunto de validación deja de mejorar, evitando así que el modelo se ajuste demasiado a los datos de entrenamiento.

**¿Cómo Funciona Early Stopping?**

Early stopping se basa en el principio de que, a medida que el modelo se entrena, el rendimiento en el conjunto de validación debería mejorar hasta cierto punto y luego comenzar a empeorar. Esto se debe a que el modelo se ajusta cada vez más a los datos de entrenamiento, lo que puede llevar a un sobreajuste. Early stopping busca detener el entrenamiento cerca del punto óptimo, antes de que el modelo comience a sobreajustarse. Es exactamente la línea punteada de las curvas de arriba.

Dos detalles de implementación importan:

- **Paciencia.** La pérdida de validación es ruidosa (sube y baja de una época a otra), así que no conviene frenar en la primera época que empeora. Se espera `patience` épocas sin mejora antes de parar. Una paciencia muy chica corta en un bache antes del mínimo real; una muy grande casi no regulariza.
- **Quedarse con los mejores pesos.** Cuando se dispara la parada, el modelo ya lleva `patience` épocas empeorando. Si no guardamos los pesos de la mejor época y los restauramos al final, nos quedamos con un modelo peor que el que vimos.


In [ ]:
class EarlyStopping:
    def __init__(self, patience=5):
        '''
        Args:
            patience (int): Cuántas épocas esperar después de la última mejora.
        '''
        # TODO: guardar patience e inicializar:
        #   counter = 0 (épocas seguidas sin mejora), best_score = inf (mejor val_loss vista),
        #   best_state = None (pesos del modelo en la mejor época), early_stop = False
        pass

    def __call__(self, val_loss, model=None):
        # TODO:
        #   - si val_loss no mejora best_score: counter += 1, y si counter >= patience -> early_stop = True
        #   - si mejora: actualizar best_score, counter = 0 y, si se pasó un modelo, guardar
        #     copy.deepcopy(model.state_dict()) en best_state (state_dict devuelve referencias: sin deepcopy se pisa)
        pass

    def restore_best(self, model):
        # TODO: si hay best_state, cargarlo en el modelo con model.load_state_dict
        pass


Nuestra clase `EarlyStopping` guarda la mejor pérdida de validación vista hasta el momento (y los pesos correspondientes) y marca `early_stop = True` si la pérdida de validación no mejora después de `patience` épocas. Lo probamos con una lista de pérdidas simulada:


In [ ]:
# lo ponemos a prueba con un ejemplo sencillo
early_stopping = EarlyStopping(patience=2)
loss_simulated = [0.1, 0.09, 0.08, 0.1, 0.11, 0.12, 0.13, 0.14, 0.15]

for i, loss in enumerate(loss_simulated):
    early_stopping(loss)
    if early_stopping.early_stop:
        print(
            f"Detener entrenamiento en la época {i + 1}, la mejor pérdida fue {early_stopping.best_score}"
        )
        break


Vamos a redefinir nuestro training loop para incluir la lógica de early stopping: es el mismo `train` de arriba, más la llamada al `EarlyStopping` después de cada época y la restauración de los mejores pesos al terminar.


In [ ]:
def train_es(
    model,
    optimizer,
    criterion,
    train_loader,
    val_loader,
    device,
    patience=5,
    epochs=10,
    log_fn=None,
    log_every=1,
):
    '''
    Entrena el modelo con early stopping y deja en el modelo los pesos de la mejor época.

    Args:
        model (torch.nn.Module): El modelo que se va a entrenar.
        optimizer (torch.optim.Optimizer): El optimizador que se utilizará para actualizar los pesos del modelo.
        criterion (torch.nn.Module): La función de pérdida que se utilizará para calcular la pérdida.
        train_loader (torch.utils.data.DataLoader): DataLoader que proporciona los datos de entrenamiento.
        val_loader (torch.utils.data.DataLoader): DataLoader que proporciona los datos de validación.
        device (str): El dispositivo donde se ejecutará el entrenamiento.
        patience (int): Número de épocas a esperar después de la última mejora en val_loss antes de detener el entrenamiento (default: 5).
        epochs (int): Número máximo de épocas de entrenamiento (default: 10).
        log_fn (function): Función que se llamará después de cada log_every épocas con los argumentos (epoch, train_loss, val_loss) (default: None).
        log_every (int): Número de épocas entre cada llamada a log_fn (default: 1).

    Returns:
        Tuple[List[float], List[float]]: Una tupla con dos listas, la primera con el error de entrenamiento de cada época y la segunda con el error de validación de cada época.

    '''
    # TODO: copiar el cuerpo de train y agregar:
    #   1. antes del loop: early_stopping = EarlyStopping(patience=patience)
    #   2. al final de cada época (después de calcular val_loss): early_stopping(val_loss, model)
    #   3. si early_stopping.early_stop: imprimir en qué época se detiene y la mejor pérdida, y salir del loop (break)
    #   4. después del loop: early_stopping.restore_best(model)
    pass


In [ ]:
torch.manual_seed(SEED)  # misma inicialización que el modelo base
ej2_model = MLP(ninputs, nclasses).to(DEVICE)

optimizer = optim.Adam(ej2_model.parameters(), lr=LR)

histories["early_stopping"] = train_es(
    model=ej2_model,
    optimizer=optimizer,
    criterion=CRITERION,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    patience=10,
    epochs=EPOCHS,
    log_fn=print_log,
    log_every=5,
)


In [ ]:
plot_training(*histories["early_stopping"], mark_min=True, title="Early stopping (patience=10)")


In [ ]:
evaluate_model("early_stopping", ej2_model, val_loader)


Tres cosas para observar:

- La curva es **idéntica** a la del modelo base hasta la época en que frena: mismo modelo, misma semilla, mismos datos. Early stopping no cambia cómo se entrena, solo cuándo se para.
- El entrenamiento frena `patience` épocas después del mínimo de validación, y gracias a `restore_best` el modelo evaluado es el de ese mínimo, no el de la última época. Es el mismo modelo que el base tenía en su mejor época, obtenido en un tercio del tiempo y **sin** las 70 épocas de memorización que siguieron.
- Con una curva más ruidosa (probar `lr=0.01`) o una paciencia muy chica (`patience=3`) el corte puede caer en un bache antes del mínimo real; la paciencia es un compromiso entre tiempo de entrenamiento y ese riesgo.


#### Lectura Adicional
- "Other than the obvious difference with the previous study (this used real data), it is
important to note another significant point: in this case. we stopped iterating (by anyone
particular criterion) when that criterion was leading to no new test set performance
improvement" - From: [Generalization and Parameter Estimation in Feedforward Nets: Some Experiments](https://proceedings.neurips.cc/paper/1989/file/63923f49e5241343aa7acb6a06a751e7-Paper.pdf)
- Prechelt, [*Early Stopping – But When?*](https://link.springer.com/chapter/10.1007/978-3-642-35289-8_5) (1998): criterios de parada y el compromiso entre paciencia y rendimiento.
- [Deep Learning Book - Regularization](https://www.deeplearningbook.org/contents/regularization.html)
- [Dive into Deep Learning - Early Stopping](https://d2l.ai/chapter_multilayer-perceptrons/generalization-deep.html#early-stopping)


### Ejercicio 3: Aplicar weight decay

**Weight decay** (o regularización L2) es la técnica de regularización más antigua de las tres. La idea: los pesos grandes permiten que la red ajuste detalles finos del conjunto de entrenamiento, incluido el ruido. Para desalentarlos, se agrega a la pérdida un término que **penaliza la magnitud de los pesos**:

$$ \mathcal{L}_{total} = \mathcal{L}_{CE} + \lambda \sum_{w} w^2 $$

Al derivar, cada peso recibe un empuje extra hacia cero proporcional a su valor. En cada paso del optimizador, además de moverse en la dirección del gradiente, el peso "decae" un poco:

$$ w \leftarrow w - \eta \left( \nabla \mathcal{L}_{CE} + 2 \lambda w \right) $$

De ahí el nombre. El coeficiente $\lambda$ controla la intensidad: con $\lambda = 0$ no hay regularización; con $\lambda$ muy grande los pesos no pueden crecer lo suficiente para aprender y el modelo subajusta.

En PyTorch no hace falta modificar la pérdida: todos los optimizadores reciben el parámetro `weight_decay`, que aplica el término dentro del paso de actualización.

> **Adam vs AdamW.** En `optim.Adam`, el término de decaimiento se suma al gradiente **antes** de la normalización adaptativa, así que su efecto depende de la escala de cada gradiente. [`optim.AdamW`](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html) ([Loshchilov & Hutter, 2019](https://arxiv.org/abs/1711.05101)) aplica el decaimiento directamente sobre los pesos, desacoplado del gradiente, y es lo que se usa hoy en la mayoría de los modelos grandes. Los valores típicos de `weight_decay` difieren mucho entre uno y otro (`1e-5` a `1e-4` en Adam, `1e-2` a `1e-1` en AdamW). Acá usamos Adam para que el único cambio respecto del modelo base sea el `weight_decay`.


In [ ]:
torch.manual_seed(SEED)  # misma inicialización que el modelo base
ej3_model = MLP(ninputs, nclasses).to(DEVICE)

optimizer = optim.Adam(ej3_model.parameters(), lr=LR, weight_decay=1e-4)  # único cambio respecto del modelo base

histories["weight_decay"] = train(
    model=ej3_model,
    optimizer=optimizer,
    criterion=CRITERION,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    epochs=EPOCHS,
    log_fn=print_log,
    log_every=5,
)


In [ ]:
plot_training(*histories["weight_decay"], mark_min=True, title="Weight decay 1e-4")


In [ ]:
evaluate_model("weight_decay", ej3_model, val_loader)


Weight decay es la única de las tres técnicas que actúa directamente sobre los pesos, así que podemos **ver** su efecto: comparamos la distribución de los pesos del modelo base y del modelo con weight decay, y la norma total.


In [ ]:
def all_weights(model):
    # concatenamos los pesos de todas las capas lineales (sin los sesgos) en un solo vector
    return torch.cat([p.detach().flatten().cpu() for n, p in model.named_parameters() if "weight" in n])


w_base = all_weights(base_model)
w_wd = all_weights(ej3_model)

print(f"Norma de los pesos | base: {w_base.norm():.1f} | weight decay: {w_wd.norm():.1f}")

plt.figure(figsize=(10, 4))
plt.hist(w_base.numpy(), bins=200, range=(-0.15, 0.15), alpha=0.6, label="base")
plt.hist(w_wd.numpy(), bins=200, range=(-0.15, 0.15), alpha=0.6, label="weight decay 1e-4")
plt.yscale("log")
plt.xlabel("valor del peso")
plt.ylabel("cantidad (escala log)")
plt.title("Distribución de los pesos")
plt.legend()
plt.show()


Con weight decay los pesos se concentran cerca de cero: la red sigue resolviendo el problema, pero con parámetros más chicos y una función más suave. En la curva, la pérdida de validación se mantiene plana hasta el final: el término de penalización impide que los pesos crezcan lo necesario para memorizar. Probar `weight_decay=1e-3`: los pesos se achican todavía más y el modelo ya no alcanza a aprender las clases minoritarias.

#### Lectura Adicional
- Krogh & Hertz, [*A Simple Weight Decay Can Improve Generalization*](https://proceedings.neurips.cc/paper/1991/hash/8eefcfdf5990e441f0fb6f3fad709e21-Abstract.html) (1992): el paper clásico.
- Loshchilov & Hutter, [*Decoupled Weight Decay Regularization*](https://arxiv.org/abs/1711.05101) (2019): por qué AdamW.
- [Dive into Deep Learning - Weight Decay](https://d2l.ai/chapter_linear-regression/weight-decay.html)


## Comparación de modelos

Juntamos lo que fuimos guardando: las curvas de validación de los cuatro modelos en una sola figura, y en la tabla las métricas macro y la pérdida de validación final de cada uno.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))

for name, (_, val_errors) in histories.items():
    axes[0].plot(val_errors, label=name)
axes[0].set_title("Pérdida de validación por época")
axes[0].set_xlabel("época")
axes[0].set_ylabel("val loss")
axes[0].legend()
axes[0].grid(True)

metrics_df = pd.DataFrame(val_metrics).T  # filas = modelos, columnas = métricas
metrics_df[["f1_macro", "recall_macro"]].plot(kind="bar", ax=axes[1], rot=0, ylim=(0.5, 1.0))
axes[1].set_title("Métricas macro en validación")
axes[1].grid(True, axis="y")

plt.tight_layout()
plt.show()

metrics_df.round(4)


Algunas observaciones para discutir en clase (los números exactos dependen de la corrida):

- **En la pérdida de validación, las tres técnicas resuelven el problema.** La curva del modelo base sube durante unas 80 épocas; las otras tres quedan planas, y la columna `val_loss` de la tabla lo cuantifica. Lo hacen por caminos distintos: dropout y weight decay cambian *qué* aprende la red; early stopping cambia *cuánto tiempo* la dejamos aprender.
- **En F1 macro la diferencia es mucho menor.** Dropout gana con claridad; early stopping y weight decay quedan a la par del base. Es lo que anticipamos al mirar la primera curva: el sobreajuste de este modelo se manifiesta sobre todo en la **confianza** de las predicciones (que es lo que mide la pérdida) y poco en las **decisiones** (que es lo que miden accuracy y F1). Con menos datos, un modelo más grande o clases más difíciles, el sobreajuste también se lleva puestas las decisiones; acá el dataset es grande y el modelo lo aprende bien de todos modos.
- **Regularizar no es gratis.** Cada técnica tiene una intensidad (`p`, `weight_decay`, `patience`) que es un hiperparámetro más. Demasiado poca no hace nada; demasiada impide aprender las clases minoritarias, que son justamente las que más pesan en el F1 macro. Probar `p=0.5` o `weight_decay=1e-3` para verlo.
- **Mirar más de una métrica.** En esta corrida el modelo base tiene el mejor *recall* macro y el peor F1 macro: los modelos regularizados predicen las clases minoritarias con más precisión pero detectan algunas menos. Cuál conviene depende del problema; en un contexto clínico, dejar pasar un latido anómalo (recall) suele costar más que una falsa alarma (precision). Es la pregunta del ejercicio 4.
- **La comparación es justa** porque los cuatro modelos parten de la misma inicialización (`torch.manual_seed(SEED)`) y se evalúan en el mismo conjunto de validación. Con `lr=0.01` (ejercicio 5) la pérdida de validación es tan ruidosa que las conclusiones cambian: vale la pena verlo.

Elegimos el modelo con mejor F1 macro **en validación** y recién ahora lo evaluamos en **test**, una sola vez. Ese es el número que reportaríamos.


In [ ]:
models = {
    "base": base_model,
    "dropout": ej1_model,
    "early_stopping": ej2_model,
    "weight_decay": ej3_model,
}

best_name = metrics_df["f1_macro"].idxmax()
print(f"Mejor modelo en validación (F1 macro): {best_name}\n")

test_labels, test_preds = model_classification_report(models[best_name], test_loader, DEVICE, TARGET_NAMES)

ConfusionMatrixDisplay.from_predictions(
    test_labels, test_preds, display_labels=["N", "S", "V", "F", "Q"], normalize="true", values_format=".2f", cmap="Blues"
)
plt.title(f"Matriz de confusión ({best_name}, test)")
plt.show()


> **A partir de la próxima clase**, `evaluate`, `train` (con early stopping incorporado), `EarlyStopping` y `model_classification_report` viven en `utils.py`, para no reescribirlos en cada notebook. La firma de `train` allí es:
>
> ```python
> train(model, optimizer, criterion, train_loader, val_loader, device,
>       do_early_stopping=True, patience=5, restore_best=True, epochs=10, log_fn=print_log, log_every=1)
> ```
